# Speech-Based Cognitive Screening — Notebook 04
## Experiment E2 · Audio Pipeline — adding the acoustic half

**Team:** A. R. Nitesh Kumar (23BAI0108) · Harshini J. S. (23BAI0166) · Hari Prasath U. (23BAI0038)  


Everything so far used the **transcripts**. This notebook adds the **acoustic** concepts you can only
hear — pitch, pitch variability, jitter, shimmer, harmonics-to-noise ratio, spectral shape, tempo,
MFCCs — then **merges them into the feature table from Notebook 01**, so Notebooks 02–03 can re-run
with sound added.

### ⭐ Before you start — two files to upload to this Colab session (left 📁 panel):
1. **`cookies.txt`** — exported from your logged-in TalkBank browser tab (the media server needs your
   browser session; username/password does **not** work on it).
2. **`Pitt.zip`** — the transcripts (gives the exact file list) and, in Cell 6, `pitt_cookie_features.csv`.

Then: **Runtime → Run all.** No GPU needed.

> **How to get `cookies.txt`:** in Chrome, install the extension *"Get cookies.txt LOCALLY"*, open a
> TalkBank page where you're logged in (e.g. the media/cookie folder), click the extension → Export.
> Upload the resulting file here.


## 0 · Setup

In [ ]:
!pip install librosa praat-parselmouth pandas pyarrow scikit-learn tqdm requests --quiet
import os, glob, re, requests, numpy as np, pandas as pd
import librosa, parselmouth
from parselmouth.praat import call
from tqdm import tqdm
print("librosa", librosa.__version__, "| parselmouth", parselmouth.__version__, "— setup OK")

## 1 · Build a session from your cookie & test it

Loads `cookies.txt` into a requests session and fetches one known file. **A status of `200` or
`206` with a large byte count means it works** (206 = "partial content", which this server uses — it
is success, not an error).


In [ ]:
import os, glob, re, requests
from tqdm import tqdm

assert os.path.exists("cookies.txt"), "Upload cookies.txt first."
BASE = "https://media.talkbank.org/dementia/English/Pitt"

def build_session(path):
    s = requests.Session()
    for line in open(path):
        if line.startswith("#") or not line.strip(): continue
        p = line.rstrip("\n").split("\t")
        if len(p) >= 7:
            s.cookies.set(p[5], p[6], domain=".talkbank.org")
    return s

sess = build_session("cookies.txt")
print("cookies loaded:", list(sess.cookies.keys()))

# STREAM the full file (fixes the 206 partial-content issue)
def fetch_full(url):
    r = sess.get(url, stream=True, timeout=60,
                 headers={"Range": "bytes=0-"})   # ask for whole file
    content = b"".join(r.iter_content(chunk_size=8192))
    return r.status_code, content

status, content = fetch_full(f"{BASE}/Dementia/cookie/007-3.mp3")
print("status:", status, "| bytes:", len(content))
if len(content) > 5000 and content[:1] != b'<':
    print("✅ cookie works — full audio downloaded. Continue.")
else:
    print("❌ still not audio — tell me the byte count.")

## 2 · Download all Cookie-Theft MP3s

Mirrors both `Control/cookie` and `Dementia/cookie` from the **noise-reduced Pitt** folder using your
cookie session. Accepts 200 **and** 206, and skips anything that is an HTML page rather than audio.
Server-to-server — nothing goes via your laptop. ~550 MB, a few minutes.


In [ ]:
# unzip transcripts for the file list
if glob.glob("Pitt.zip") and not glob.glob("**/Dementia/cookie/*.cha", recursive=True):
    os.system("unzip -o Pitt.zip -d . > /dev/null 2>&1")

ids = []
for grp in ["Control", "Dementia"]:
    for f in glob.glob(f"**/{grp}/cookie/*.cha", recursive=True):
        ids.append((grp, os.path.basename(f)[:-4]))
if not ids:
    for grp in ["Control", "Dementia"]:
        rr = sess.get(f"{BASE}/{grp}/cookie/")
        for m in re.findall(r'href="([^"]+\.mp3)"', rr.text):
            ids.append((grp, m[:-4]))
ids = sorted(set(ids))
print(f"Downloading {len(ids)} recordings …")

os.makedirs("audio", exist_ok=True)
ok, fail = 0, []
for grp, fid in tqdm(ids):
    out = f"audio/{grp}/{fid}.mp3"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    if os.path.exists(out) and os.path.getsize(out) > 5000:
        ok += 1; continue
    try:
        r = sess.get(f"{BASE}/{grp}/cookie/{fid}.mp3", stream=True, timeout=60,
                     headers={"Range": "bytes=0-"})
        content = b"".join(r.iter_content(chunk_size=8192))
        if len(content) > 5000 and content[:1] != b'<':
            open(out, "wb").write(content); ok += 1
        else:
            fail.append((fid, r.status_code, len(content)))
    except Exception as e:
        fail.append((fid, "ERR", str(e)[:30]))

print(f"\n✅ downloaded {ok} MP3s")
if fail: print(f"⚠️ {len(fail)} failed — first few: {fail[:5]}")
sz = sum(os.path.getsize(f) for f in glob.glob("audio/**/*.mp3", recursive=True))
print(f"total audio size: {sz/1e6:.0f} MB")

## 3 · Acoustic feature extractor

Clinical voice markers via Praat (pitch, jitter, shimmer, HNR) and spectral/tempo markers via
librosa. *(Validated on a 140 Hz synthetic vowel: f0_mean = 140.0 with sensible jitter/shimmer/HNR.)*


In [ ]:
def acoustic_features(y, sr):
    f = {}
    try:
        snd = parselmouth.Sound(y.astype(np.float64), sampling_frequency=sr)
        pitch = snd.to_pitch()
        f0v = pitch.selected_array['frequency']; f0v = f0v[f0v > 0]
        f["f0_mean"]  = float(np.mean(f0v)) if len(f0v) else np.nan
        f["f0_std"]   = float(np.std(f0v))  if len(f0v) else np.nan
        f["f0_range"] = float(np.ptp(f0v))  if len(f0v) else np.nan
        pp = call(snd, "To PointProcess (periodic, cc)", 75, 500)
        f["jitter_local"]  = call(pp, "Get jitter (local)", 0,0,1e-4,0.02,1.3)
        f["shimmer_local"] = call([snd,pp], "Get shimmer (local)", 0,0,1e-4,0.02,1.3,1.6)
        harm = call(snd, "To Harmonicity (cc)", 0.01,75,0.1,1.0)
        f["hnr"] = call(harm, "Get mean", 0,0)
    except Exception:
        for kk in ["f0_mean","f0_std","f0_range","jitter_local","shimmer_local","hnr"]:
            f.setdefault(kk, np.nan)
    f["spec_centroid"]  = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    f["spec_bandwidth"] = float(np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)))
    f["zcr"]            = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    f["rms_energy"]     = float(np.mean(librosa.feature.rms(y=y)))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        f[f"mfcc{i+1}"] = float(np.mean(mfcc[i]))
    return f
print("extractor ready")

## 4 · Extract features over every MP3

~552 short clips take a few minutes on CPU. Output is keyed by the same `file` id as the transcripts.


In [ ]:
rows = []
mp3s = sorted(glob.glob("audio/**/*.mp3", recursive=True))
assert mp3s, "No MP3s found — complete Cell 2 first."

for path in tqdm(mp3s):
    fid = os.path.basename(path)[:-4]
    try:
        y, sr = librosa.load(path, sr=16000, mono=True)
        feats = acoustic_features(y, sr)
        feats["file"] = fid
        feats["audio_dur_s"] = len(y)/sr
        rows.append(feats)
    except Exception as e:
        print("skip", fid, e)

audio_df = pd.DataFrame(rows)
audio_df.to_parquet("pitt_cookie_audio_features.parquet", index=False)
print(f"\nextracted acoustic features for {len(audio_df)} recordings")
audio_df.head()

## 5 · Merge audio + text

Upload **`pitt_cookie_features.csv`** from Notebook 01 when asked. Joins on the `file` id; recordings
without audio get NaN (handled downstream). This merged table is the new input for Notebooks 02–03.


In [ ]:
tcand = [c for c in glob.glob("**/pitt_cookie_features.*", recursive=True)
         if "AUDIO" not in c and "audio" not in c]
if not tcand:
    try:
        from google.colab import files
        print("Upload pitt_cookie_features.csv from Notebook 01 …")
        files.upload()
        tcand = [c for c in glob.glob("**/pitt_cookie_features.*", recursive=True)
                 if "AUDIO" not in c and "audio" not in c]
    except Exception as e:
        print(e)
tpath = ([c for c in tcand if c.endswith(".parquet")] or tcand)[0]
text_df = pd.read_parquet(tpath) if tpath.endswith(".parquet") else pd.read_csv(tpath)

merged = text_df.merge(audio_df, on="file", how="left")
merged.to_parquet("pitt_cookie_features_AUDIO.parquet", index=False)
merged.to_csv("pitt_cookie_features_AUDIO.csv", index=False)

n_audio = merged["f0_mean"].notna().sum()
print(f"merged: {merged.shape} · acoustic features present for {n_audio}/{len(merged)} recordings")
print("saved -> pitt_cookie_features_AUDIO.csv")

## 6 · Does the voice add to the words?

The E2 result: baseline classifier on **text-only** vs **audio-only** vs **text+audio**, same
speaker-independent CV. If fused beats text, acoustic information is genuinely additive.


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

d = merged[merged.dx.isin(["Control","ProbableAD"])].copy()
d["label"] = (d.dx=="ProbableAD").astype(int)

TEXT = ["mattr","pronoun_noun_ratio","mlu_words","lexical_density","filled_pause_rate",
        "speech_rate_wpm","long_pause_ratio","mean_pause_ms","ttr","noun_verb_ratio"]
AUDIO = ["f0_mean","f0_std","f0_range","jitter_local","shimmer_local","hnr",
         "spec_centroid","spec_bandwidth","zcr","rms_energy"] + [f"mfcc{i}" for i in range(1,14)]
TEXT  = [c for c in TEXT  if c in d.columns]
AUDIO = [c for c in AUDIO if c in d.columns]

def cv_auc(cols):
    sub = d.dropna(subset=cols, how="all")
    X=sub[cols].values; y=sub.label.values; g=sub.participant_id.values
    sgkf=StratifiedGroupKFold(5,shuffle=True,random_state=42); a=[]
    for tr,te in sgkf.split(X,y,g):
        assert not(set(g[tr])&set(g[te]))
        clf=make_pipeline(SimpleImputer(),StandardScaler(),
                          RandomForestClassifier(n_estimators=300,random_state=0))
        clf.fit(X[tr],y[tr]); a.append(roc_auc_score(y[te],clf.predict_proba(X[te])[:,1]))
    return np.mean(a),np.std(a),len(sub)

t_mu,t_sd,nt = cv_auc(TEXT)
a_mu,a_sd,na = cv_auc(AUDIO)
f_mu,f_sd,nf = cv_auc(TEXT+AUDIO)
print(f"Text only    AUC {t_mu:.3f} ± {t_sd:.3f}   (n={nt})")
print(f"Audio only   AUC {a_mu:.3f} ± {a_sd:.3f}   (n={na})")
print(f"Text + Audio AUC {f_mu:.3f} ± {f_sd:.3f}   (n={nf})")
print(f"\n>>> Audio adds {f_mu-t_mu:+.3f} AUC over text alone <<<")

## 7 · What you produced & what's next

- **`pitt_cookie_features_AUDIO.csv`** — the enlarged table (text + acoustic concepts).
- A **text vs audio vs fused** result.

### Plug the richer table into the earlier notebooks
1. **Notebook 02** — point the loader at `pitt_cookie_features_AUDIO.csv`, add the acoustic columns
   to `CONCEPTS`, re-run the interpretability-cost comparison.
2. **Notebook 03** — same swap; the XAI audit now judges faithfulness on **acoustic** concepts too.
3. **Optional — Grad-CAM + spectrogram CNN:** the imaging-style visual explanation from the design doc.
4. **Optional — wav2vec 2.0:** pooled self-supervised embeddings for the black-box baseline.

> Report every metric as **mean ± std over folds/seeds**, not a single run.
